# DICOM Header Extraction Demo: Three Output Methods

This notebook demonstrates three ways to extract DICOM header metadata using `dicom_to_parquet.py`.  
You can run any single method independently.

| Method | Description | Output |
|--------|-------------|--------|
| **Method 1** | Parquet only | Hive-partitioned Parquet (`modality/tag/`) |
| **Method 2** | Parquet → CSV | Parquet first, then merge all partitions to a single CSV |
| **Method 3** | CSV only | Write a single flat CSV directly (no Parquet) |

**Steps**
1. Run the **[Common] Imports** cell below once.
2. Jump to the method you want, edit the path config cell, then run its cells in order.

In [7]:
# ════════════════════════════════════════════════
# [Common] Imports — run this cell first regardless of which method you choose
# ════════════════════════════════════════════════

import os, sys

# Change working directory to the folder containing dicom_to_parquet.py
SCRIPT_DIR = r"C:\Projects\dicom_heterogeneity\dicom_to_parquet"
%cd $SCRIPT_DIR
sys.path.insert(0, SCRIPT_DIR)

from dicom_to_parquet import (
    build_parquet,
    build_csv_direct,
    merge_to_csv,
    iter_dicom_files,
    ARROW_SCHEMA,
)

import pyarrow.dataset as pad
import pandas as pd
from IPython.display import display

print("Imports OK")

C:\Projects\dicom_heterogeneity\dicom_to_parquet
Imports OK


---
## Method 1: Parquet Only

Writes DICOM headers to a **Hive-partitioned Parquet dataset** (`modality=.../tag=.../part-b0-0.parquet`).

- Best choice for large datasets — only the partitions you query are read
- Compatible with pyarrow, DuckDB, Spark, etc.
- Can be converted to CSV later with `merge_to_csv()` if needed

In [8]:
# ════════════════════════════════════════════════
# [Method 1] Path & option config — edit this cell
# ════════════════════════════════════════════════

# Input: top-level folder containing .dcm files (searched recursively)
M1_DICOM_ROOT = r"C:\Projects\dicom_data\data\mimic-iv-echo"

# Output: folder where Parquet partitions will be written
M1_OUT_DIR    = r"C:\Projects\dicom_data\data\mimic-iv-echo_headers\01_parquet_only"

# Options
M1_SHARD_ROWS      = 2_000_000  # max rows per Parquet file within a partition
M1_SKIP_PIXEL_DATA = True       # exclude (7FE0,0010) Pixel Data — recommended
M1_EXTRA_SKIP_TAGS = []         # additional tags to skip, e.g. ['00209999']
M1_VERBOSE_EVERY   = 1          # print progress every N files (use 1000+ for large datasets)

# Preview input
dcm_files = list(iter_dicom_files(M1_DICOM_ROOT))
print(f"DICOM files : {len(dcm_files)}")
print(f"Output dir  : {M1_OUT_DIR}")

DICOM files : 5
Output dir  : C:\Projects\dicom_data\data\mimic-iv-echo_headers\01_parquet_only


In [9]:
# [Method 1] Run
build_parquet(
    dicom_root      = M1_DICOM_ROOT,
    out_dir         = M1_OUT_DIR,
    max_files       = None,
    shard_rows      = M1_SHARD_ROWS,
    skip_pixel_data = M1_SKIP_PIXEL_DATA,
    extra_skip_tags = M1_EXTRA_SKIP_TAGS,
    verbose_every   = M1_VERBOSE_EVERY,
    export_csv      = None,
)

[INFO] files=1 bad=0 rows_buffer=62 rows_total=62
[INFO] files=2 bad=0 rows_buffer=136 rows_total=136
[INFO] files=3 bad=0 rows_buffer=215 rows_total=215
[INFO] files=4 bad=0 rows_buffer=289 rows_total=289
[INFO] files=5 bad=0 rows_buffer=351 rows_total=351
[DONE]
{
  "dicom_root": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo",
  "out_dir": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo_headers\\01_parquet_only",
  "files_processed": 5,
  "files_failed": 0,
  "rows_total": 351,
  "batches_written": 1,
  "skip_pixel_data": true,
  "skip_tags": [
    "54001010",
    "7FE00010"
  ],
  "compression": "zstd",
  "schema": [
    "file_path:string",
    "study_uid:string",
    "series_uid:string",
    "instance_uid:string",
    "sop_class_uid:string",
    "modality:string",
    "tag:string",
    "vr:string",
    "vm:string",
    "path:string",
    "value:string"
  ]
}


In [10]:
# [Method 1] Inspect results

dataset1 = pad.dataset(M1_OUT_DIR, format="parquet", partitioning="hive")
df1 = dataset1.to_table().to_pandas()

print(f"Total rows     : {len(df1):,}")
print(f"Unique tags    : {df1['tag'].nunique():,}")
print(f"Modalities     : {sorted(df1['modality'].dropna().unique())}")

# Verify private tags are present (group number is odd)
priv1 = df1[df1['tag'].apply(lambda t: int(t[:4], 16) % 2 == 1)]
print(f"Private tag rows: {len(priv1):,}")
if not priv1.empty:
    display(priv1[['tag','vr','path','value']].drop_duplicates('tag').head(5))

print("\n-- Data preview (first 10 rows) --")
display(df1.head(10))

print("\n-- Partition files written --")
for root, _, files in os.walk(M1_OUT_DIR):
    for f in sorted(files):
        full = os.path.join(root, f)
        size_kb = os.path.getsize(full) / 1024
        rel = os.path.relpath(full, M1_OUT_DIR)
        print(f"  {rel:55s}  {size_kb:6.1f} KB")

Total rows     : 351
Unique tags    : 79
Modalities     : ['US']
Private tag rows: 0

-- Data preview (first 10 rows) --


,file_path,study_uid,series_uid,instance_uid,sop_class_uid,vr,vm,path,value,modality,tag
0,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
1,C:\Projects\dicom_data\data\mimic-iv-echo\9410...,1.2.840.113554.6.1.104.68301667710199290520777...,1.2.840.113554.6.1.105.52875607447023178069717...,1.2.840.113554.6.1.101.69855423628295450049911...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.3.1,US,00080016
2,C:\Projects\dicom_data\data\mimic-iv-echo\9802...,1.2.840.113554.6.1.104.12917773695659117017448...,1.2.840.113554.6.1.105.75593455491512733255625...,1.2.840.113554.6.1.101.59351117373448806027346...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
3,C:\Projects\dicom_data\data\mimic-iv-echo\9832...,1.2.840.113554.6.1.104.63742685390757424135156...,1.2.840.113554.6.1.105.97320391386664344724944...,1.2.840.113554.6.1.101.27493956172437740663967...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.3.1,US,00080016
4,C:\Projects\dicom_data\data\mimic-iv-echo\9989...,1.2.840.113554.6.1.104.86871504177171794777507...,1.2.840.113554.6.1.105.64352159466407455443550...,1.2.840.113554.6.1.101.12795021409745600246926...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
5,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.50408064018308145405200...,US,00080018
6,C:\Projects\dicom_data\data\mimic-iv-echo\9410...,1.2.840.113554.6.1.104.68301667710199290520777...,1.2.840.113554.6.1.105.52875607447023178069717...,1.2.840.113554.6.1.101.69855423628295450049911...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.69855423628295450049911...,US,00080018
7,C:\Projects\dicom_data\data\mimic-iv-echo\9802...,1.2.840.113554.6.1.104.12917773695659117017448...,1.2.840.113554.6.1.105.75593455491512733255625...,1.2.840.113554.6.1.101.59351117373448806027346...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.59351117373448806027346...,US,00080018
8,C:\Projects\dicom_data\data\mimic-iv-echo\9832...,1.2.840.113554.6.1.104.63742685390757424135156...,1.2.840.113554.6.1.105.97320391386664344724944...,1.2.840.113554.6.1.101.27493956172437740663967...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.27493956172437740663967...,US,00080018
9,C:\Projects\dicom_data\data\mimic-iv-echo\9989...,1.2.840.113554.6.1.104.86871504177171794777507...,1.2.840.113554.6.1.105.64352159466407455443550...,1.2.840.113554.6.1.101.12795021409745600246926...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.12795021409745600246926...,US,00080018



-- Partition files written --
  _build_summary.json                                         0.6 KB
  modality=US\tag=00080016\part-b0-0.parquet                  4.0 KB
  modality=US\tag=00080018\part-b0-0.parquet                  4.3 KB
  modality=US\tag=00080020\part-b0-0.parquet                  4.0 KB
  modality=US\tag=00080021\part-b0-0.parquet                  4.0 KB
  modality=US\tag=00080023\part-b0-0.parquet                  4.0 KB
  modality=US\tag=0008002A\part-b0-0.parquet                  4.0 KB
  modality=US\tag=00080030\part-b0-0.parquet                  3.9 KB
  modality=US\tag=00080031\part-b0-0.parquet                  3.9 KB
  modality=US\tag=00080033\part-b0-0.parquet                  4.0 KB
  modality=US\tag=00080050\part-b0-0.parquet                  3.9 KB
  modality=US\tag=00080060\part-b0-0.parquet                  3.9 KB
  modality=US\tag=00080070\part-b0-0.parquet                  4.0 KB
  modality=US\tag=00080080\part-b0-0.parquet                  4.0 KB
  m

---
## Method 2: Parquet, then Merge to CSV

Writes Parquet partitions first (same as Method 1), then **merges all partitions into a single flat CSV**.

- Produces two outputs: the Parquet dataset and `all_tags.csv`
- If you already have a Parquet dataset, you can call `merge_to_csv()` standalone
- (caution) The CSV merge step can be very slow and produce a very large file for large datasets

In [11]:
# ════════════════════════════════════════════════
# [Method 2] Path & option config — edit this cell
# ════════════════════════════════════════════════

# Input: top-level folder containing .dcm files
M2_DICOM_ROOT = r"C:\Projects\dicom_data\data\mimic-iv-echo"

# Output 1: folder for Parquet partitions
M2_OUT_DIR    = r"C:\Projects\dicom_data\data\mimic-iv-echo_headers\02_parquet_and_csv"

# Output 2: merged CSV path (auto-created after Parquet is done)
M2_EXPORT_CSV = r"C:\Projects\dicom_data\data\mimic-iv-echo_headers\02_parquet_and_csv\all_tags.csv"

# Options
M2_SHARD_ROWS      = 2_000_000
M2_SKIP_PIXEL_DATA = True
M2_EXTRA_SKIP_TAGS = []
M2_VERBOSE_EVERY   = 1

dcm_files = list(iter_dicom_files(M2_DICOM_ROOT))
print(f"DICOM files  : {len(dcm_files)}")
print(f"Parquet dir  : {M2_OUT_DIR}")
print(f"CSV output   : {M2_EXPORT_CSV}")

DICOM files  : 5
Parquet dir  : C:\Projects\dicom_data\data\mimic-iv-echo_headers\02_parquet_and_csv
CSV output   : C:\Projects\dicom_data\data\mimic-iv-echo_headers\02_parquet_and_csv\all_tags.csv


In [12]:
# [Method 2] Run: write Parquet partitions, then merge to CSV
build_parquet(
    dicom_root      = M2_DICOM_ROOT,
    out_dir         = M2_OUT_DIR,
    max_files       = None,
    shard_rows      = M2_SHARD_ROWS,
    skip_pixel_data = M2_SKIP_PIXEL_DATA,
    extra_skip_tags = M2_EXTRA_SKIP_TAGS,
    verbose_every   = M2_VERBOSE_EVERY,
    export_csv      = M2_EXPORT_CSV,   # triggers CSV merge after Parquet is complete
)

[INFO] files=1 bad=0 rows_buffer=62 rows_total=62
[INFO] files=2 bad=0 rows_buffer=136 rows_total=136
[INFO] files=3 bad=0 rows_buffer=215 rows_total=215
[INFO] files=4 bad=0 rows_buffer=289 rows_total=289
[INFO] files=5 bad=0 rows_buffer=351 rows_total=351
[DONE]
{
  "dicom_root": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo",
  "out_dir": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo_headers\\02_parquet_and_csv",
  "files_processed": 5,
  "files_failed": 0,
  "rows_total": 351,
  "batches_written": 1,
  "skip_pixel_data": true,
  "skip_tags": [
    "54001010",
    "7FE00010"
  ],
  "compression": "zstd",
  "schema": [
    "file_path:string",
    "study_uid:string",
    "series_uid:string",
    "instance_uid:string",
    "sop_class_uid:string",
    "modality:string",
    "tag:string",
    "vr:string",
    "vm:string",
    "path:string",
    "value:string"
  ]
}
[CSV] Scanning partitions in C:\Projects\dicom_data\data\mimic-iv-echo_headers\02_parquet_and_csv ...
[CSV] Done: 351 rows

[CSV] 5 rows written ...
[CSV] 10 rows written ...
[CSV] 15 rows written ...
[CSV] 20 rows written ...
[CSV] 25 rows written ...
[CSV] 30 rows written ...
[CSV] 35 rows written ...
[CSV] 40 rows written ...
[CSV] 45 rows written ...
[CSV] 50 rows written ...
[CSV] 55 rows written ...
[CSV] 60 rows written ...
[CSV] 65 rows written ...
[CSV] 70 rows written ...
[CSV] 75 rows written ...
[CSV] 80 rows written ...
[CSV] 81 rows written ...
[CSV] 86 rows written ...
[CSV] 91 rows written ...
[CSV] 93 rows written ...
[CSV] 95 rows written ...
[CSV] 97 rows written ...
[CSV] 102 rows written ...
[CSV] 107 rows written ...
[CSV] 112 rows written ...
[CSV] 117 rows written ...
[CSV] 122 rows written ...
[CSV] 127 rows written ...
[CSV] 132 rows written ...
[CSV] 137 rows written ...
[CSV] 139 rows written ...
[CSV] 141 rows written ...
[CSV] 146 rows written ...
[CSV] 148 rows written ...
[CSV] 150 rows written ...
[CSV] 155 rows written ...
[CSV] 157 rows written ...
[CSV] 159 rows written .

In [13]:
# [Method 2] Inspect results

# Parquet (exclude_invalid_files=True: ignore non-parquet files such as all_tags.csv in the same folder)
dataset2 = pad.dataset(M2_OUT_DIR, format="parquet", partitioning="hive", exclude_invalid_files=True)
df2_parquet = dataset2.to_table().to_pandas()
print(f"[Parquet] rows: {len(df2_parquet):,}")

# CSV
df2_csv = pd.read_csv(M2_EXPORT_CSV)
csv2_size_mb = os.path.getsize(M2_EXPORT_CSV) / 1024 / 1024
print(f"[CSV]     rows: {len(df2_csv):,}")
print(f"[CSV]     size: {csv2_size_mb:.3f} MB")

if len(df2_parquet) == len(df2_csv):
    print("OK: Parquet row count == CSV row count")
else:
    print(f"WARNING: row count mismatch — Parquet {len(df2_parquet):,} / CSV {len(df2_csv):,}")

print("\n-- CSV preview (first 10 rows) --")
display(df2_csv.head(10))

print("\n-- Files written --")
for root, _, files in os.walk(M2_OUT_DIR):
    for f in sorted(files):
        full = os.path.join(root, f)
        size_kb = os.path.getsize(full) / 1024
        rel = os.path.relpath(full, M2_OUT_DIR)
        print(f"  {rel:55s}  {size_kb:6.1f} KB")

[Parquet] rows: 351
[CSV]     rows: 351
[CSV]     size: 0.114 MB
OK: Parquet row count == CSV row count

-- CSV preview (first 10 rows) --


,file_path,study_uid,series_uid,instance_uid,sop_class_uid,vr,vm,path,value,modality,tag
0,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
1,C:\Projects\dicom_data\data\mimic-iv-echo\9410...,1.2.840.113554.6.1.104.68301667710199290520777...,1.2.840.113554.6.1.105.52875607447023178069717...,1.2.840.113554.6.1.101.69855423628295450049911...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.3.1,US,00080016
2,C:\Projects\dicom_data\data\mimic-iv-echo\9802...,1.2.840.113554.6.1.104.12917773695659117017448...,1.2.840.113554.6.1.105.75593455491512733255625...,1.2.840.113554.6.1.101.59351117373448806027346...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
3,C:\Projects\dicom_data\data\mimic-iv-echo\9832...,1.2.840.113554.6.1.104.63742685390757424135156...,1.2.840.113554.6.1.105.97320391386664344724944...,1.2.840.113554.6.1.101.27493956172437740663967...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.3.1,US,00080016
4,C:\Projects\dicom_data\data\mimic-iv-echo\9989...,1.2.840.113554.6.1.104.86871504177171794777507...,1.2.840.113554.6.1.105.64352159466407455443550...,1.2.840.113554.6.1.101.12795021409745600246926...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
5,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.50408064018308145405200...,US,00080018
6,C:\Projects\dicom_data\data\mimic-iv-echo\9410...,1.2.840.113554.6.1.104.68301667710199290520777...,1.2.840.113554.6.1.105.52875607447023178069717...,1.2.840.113554.6.1.101.69855423628295450049911...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.69855423628295450049911...,US,00080018
7,C:\Projects\dicom_data\data\mimic-iv-echo\9802...,1.2.840.113554.6.1.104.12917773695659117017448...,1.2.840.113554.6.1.105.75593455491512733255625...,1.2.840.113554.6.1.101.59351117373448806027346...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.59351117373448806027346...,US,00080018
8,C:\Projects\dicom_data\data\mimic-iv-echo\9832...,1.2.840.113554.6.1.104.63742685390757424135156...,1.2.840.113554.6.1.105.97320391386664344724944...,1.2.840.113554.6.1.101.27493956172437740663967...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.27493956172437740663967...,US,00080018
9,C:\Projects\dicom_data\data\mimic-iv-echo\9989...,1.2.840.113554.6.1.104.86871504177171794777507...,1.2.840.113554.6.1.105.64352159466407455443550...,1.2.840.113554.6.1.101.12795021409745600246926...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.12795021409745600246926...,US,00080018



-- Files written --
  _build_summary.json                                         0.6 KB
  all_tags.csv                                              116.9 KB
  modality=US\tag=00080016\part-b0-0.parquet                  4.0 KB
  modality=US\tag=00080018\part-b0-0.parquet                  4.3 KB
  modality=US\tag=00080020\part-b0-0.parquet                  4.0 KB
  modality=US\tag=00080021\part-b0-0.parquet                  4.0 KB
  modality=US\tag=00080023\part-b0-0.parquet                  4.0 KB
  modality=US\tag=0008002A\part-b0-0.parquet                  4.0 KB
  modality=US\tag=00080030\part-b0-0.parquet                  3.9 KB
  modality=US\tag=00080031\part-b0-0.parquet                  3.9 KB
  modality=US\tag=00080033\part-b0-0.parquet                  4.0 KB
  modality=US\tag=00080050\part-b0-0.parquet                  3.9 KB
  modality=US\tag=00080060\part-b0-0.parquet                  3.9 KB
  modality=US\tag=00080070\part-b0-0.parquet                  4.0 KB
  modality=US

---
## Method 3: CSV Only (no Parquet)

Writes all DICOM headers directly to a **single flat CSV** without creating any Parquet files.

- Uses the same tag-explosion logic as Methods 1 & 2: private tags, nested SQ, multi-value all included
- Useful when Parquet tooling is unavailable or the dataset is small
- (caution) Can produce a very large file — Method 1 is recommended for large datasets

In [14]:
# ════════════════════════════════════════════════
# [Method 3] Path & option config — edit this cell
# ════════════════════════════════════════════════

# Input: top-level folder containing .dcm files
M3_DICOM_ROOT = r"C:\Projects\dicom_data\data\mimic-iv-echo"

# Output: CSV file path (including filename)
M3_CSV_PATH   = r"C:\Projects\dicom_data\data\mimic-iv-echo_headers\03_csv_only\all_tags.csv"

# Options
M3_SKIP_PIXEL_DATA = True
M3_EXTRA_SKIP_TAGS = []
M3_VERBOSE_EVERY   = 1
M3_BUFFER_ROWS     = 100_000  # flush to disk every N rows to keep memory usage low

os.makedirs(os.path.dirname(M3_CSV_PATH), exist_ok=True)

dcm_files = list(iter_dicom_files(M3_DICOM_ROOT))
print(f"DICOM files : {len(dcm_files)}")
print(f"CSV output  : {M3_CSV_PATH}")

DICOM files : 5
CSV output  : C:\Projects\dicom_data\data\mimic-iv-echo_headers\03_csv_only\all_tags.csv


In [15]:
# [Method 3] Run: write CSV directly (no intermediate Parquet)
build_csv_direct(
    dicom_root      = M3_DICOM_ROOT,
    csv_path        = M3_CSV_PATH,
    max_files       = None,
    skip_pixel_data = M3_SKIP_PIXEL_DATA,
    extra_skip_tags = M3_EXTRA_SKIP_TAGS,
    verbose_every   = M3_VERBOSE_EVERY,
    buffer_rows     = M3_BUFFER_ROWS,
)

[INFO] files=1 bad=0 rows=62
[INFO] files=2 bad=0 rows=136
[INFO] files=3 bad=0 rows=215
[INFO] files=4 bad=0 rows=289
[INFO] files=5 bad=0 rows=351
[DONE]
{
  "dicom_root": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo",
  "csv_path": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo_headers\\03_csv_only\\all_tags.csv",
  "files_processed": 5,
  "files_failed": 0,
  "rows_total": 351
}


In [16]:
# [Method 3] Inspect results

df3 = pd.read_csv(M3_CSV_PATH)
csv3_size_mb = os.path.getsize(M3_CSV_PATH) / 1024 / 1024

print(f"Total rows      : {len(df3):,}")
print(f"File size       : {csv3_size_mb:.3f} MB")
print(f"Unique tags     : {df3['tag'].nunique():,}")

# Verify private tags
priv3 = df3[df3['tag'].apply(lambda t: int(t[:4], 16) % 2 == 1)]
print(f"Private tag rows: {len(priv3):,}")
if not priv3.empty:
    display(priv3[['tag','vr','path','value']].drop_duplicates('tag').head(5))

# Verify nested SQ expansion (path contains '[N]' = inside a sequence item)
sq3 = df3[df3['path'].str.contains(r'\[\d+\]', regex=True)]
print(f"Nested SQ rows  : {len(sq3):,}")
if not sq3.empty:
    display(sq3[['tag','vr','path','value']].head(5))

print("\n-- CSV preview (first 10 rows) --")
display(df3.head(10))

Total rows      : 351
File size       : 0.114 MB
Unique tags     : 79
Private tag rows: 0
Nested SQ rows  : 346


,tag,vr,path,value
0,00080016,UI,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1
1,00080018,UI,00080018[0]/,1.2.840.113554.6.1.101.50408064018308145405200...
2,00080020,DA,00080020[0]/,21510406
3,00080021,DA,00080021[0]/,21510406
4,00080023,DA,00080023[0]/,21510406



-- CSV preview (first 10 rows) --


,file_path,study_uid,series_uid,instance_uid,sop_class_uid,modality,tag,vr,vm,path,value
0,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080016,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1
1,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080018,UI,1,00080018[0]/,1.2.840.113554.6.1.101.50408064018308145405200...
2,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080020,DA,1,00080020[0]/,21510406
3,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080021,DA,1,00080021[0]/,21510406
4,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080023,DA,1,00080023[0]/,21510406
5,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,0008002A,DT,1,0008002A[0]/,21510406083003
6,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080030,TM,1,00080030[0]/,081600
7,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080031,TM,1,00080031[0]/,081600
8,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080033,TM,1,00080033[0]/,083003
9,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080050,SH,0,00080050[0]/,NaN


---
## [Optional] Compare All Three Methods

Run this section only after all three methods have been executed.

In [17]:
# [Optional] Side-by-side comparison of all three methods
# Requires df1, df2_csv, df3 to be defined (i.e. all three methods were run)

def folder_size_mb(path):
    total = sum(
        os.path.getsize(os.path.join(r, f))
        for r, _, files in os.walk(path)
        for f in files
    )
    return total / 1024 / 1024

def count_private(df):
    return len(df[df['tag'].apply(lambda t: int(t[:4], 16) % 2 == 1)])

summary = pd.DataFrame([
    {
        "Method": "1: Parquet only",
        "Total rows": len(df1),
        "Unique tags": df1['tag'].nunique(),
        "Private tag rows": count_private(df1),
        "Disk size (MB)": round(folder_size_mb(M1_OUT_DIR), 3),
        "Output type": "Partitioned Parquet",
    },
    {
        "Method": "2: Parquet + CSV",
        "Total rows": len(df2_csv),
        "Unique tags": df2_csv['tag'].nunique(),
        "Private tag rows": count_private(df2_csv),
        "Disk size (MB)": round(folder_size_mb(M2_OUT_DIR), 3),
        "Output type": "Partitioned Parquet + CSV",
    },
    {
        "Method": "3: CSV only",
        "Total rows": len(df3),
        "Unique tags": df3['tag'].nunique(),
        "Private tag rows": count_private(df3),
        "Disk size (MB)": round(folder_size_mb(os.path.dirname(M3_CSV_PATH)), 3),
        "Output type": "Single CSV",
    },
])

display(summary.set_index("Method"))

counts = [len(df1), len(df2_csv), len(df3)]
if len(set(counts)) == 1:
    print(f"\nOK: all three methods produced the same row count ({counts[0]:,} rows)")
else:
    print(f"\nWARNING: row count mismatch across methods: {counts}")

,Total rows,Unique tags,Private tag rows,Disk size (MB),Output type
Method,,,,,
1: Parquet only,351,79,0,0.302,Partitioned Parquet
2: Parquet + CSV,351,79,0,0.416,Partitioned Parquet + CSV
3: CSV only,351,79,0,0.114,Single CSV



OK: all three methods produced the same row count (351 rows)


In [18]:
# [Optional] Tag distribution analysis (uses df1 from Method 1)

print("-- VR distribution --")
display(df1['vr'].value_counts().rename_axis('VR').reset_index(name='count').head(15))

print("\n-- Top 20 most frequent tags --")
top_tags = (
    df1.groupby('tag')
    .agg(count=('tag','count'), VR=('vr','first'), sample_value=('value','first'))
    .sort_values('count', ascending=False)
    .head(20)
)
display(top_tags)

-- VR distribution --


,VR,count
0,US,68
1,LO,41
2,UL,38
3,CS,33
4,IS,27
5,DS,21
6,UI,20
7,DA,20
8,TM,15
9,PN,15



-- Top 20 most frequent tags --


,count,VR,sample_value
tag,,,
0018601E,6,UL,519
0018601C,6,UL,796
00186022,6,SL,1
0018602E,6,FD,0.0359764
00186024,6,US,3
00186026,6,US,3
0018602C,6,FD,0.0359764
00186020,6,SL,289
00186016,6,UL,0
